# 01 — Quick Start

This notebook walks through the basic `quant_lib` workflow:
list available experiments, run exploration, inspect output.

**Prerequisites:** `pip install -e .` and a configured `.env` with `QUANT_LIB_HMAC_SECRET` (run `quant_exp init`).

## Setup

In [ ]:
import quant_lib
print(f"quant_lib version: {quant_lib.__version__}")

## List available experiments

In [ ]:
from quant_lib.cli.list_cmd import list_cmd
list_cmd()  # prints all registered experiments

## Run exploration (Phase 0-3)

In [ ]:
# Run the exploration phase: fetch data, compute features,
# walk-forward analysis, SPA p-value, per-symbol risk allocation.
#
# Post Hansen-literal SPA (claim #3 Blocker A fix):
#  - ``result`` is an ``ExploreResult`` dataclass (Sprint 3 fix
#    3.3). 8 fields: ``experiment``, ``n_oos_trades``, ``n_executed``,
#    ``n_rejected``, ``final_equity``, ``spa_p_value``,
#    ``spa_naive_p_value``, ``narrowed_symbols``. ``spa_p_value``
#    now carries the Hansen-corrected p when WFA ``trial_r_nets``
#    are available (NaN-safe fallback to legacy p); the legacy
#    circular-permutation p is preserved in ``spa_naive_p_value``
#    for transparency. There is no ``stage`` / ``eligible_symbols``
#    attribute on ExploreResult (those are Candidate-only
#    legacy fields).
result = quant_lib.run_explore(
    experiment_name="vol_compression_v1",
    cache_dir="./data_cache",
)
print(f"Experiment: {result.experiment}")
print(f"Narrowed symbols: {result.narrowed_symbols}")
print(f"SPA p-value (Hansen when trial_r_nets available, "
      f"fallback to legacy): {result.spa_p_value:.4f}")
print(f"Legacy SPA p-value (preserved for transparency): "
      f"{result.spa_naive_p_value}")

## Inspect result

In [ ]:
# ``edge_metrics`` is a flat TOP-LEVEL dict (Sprint 3 fix 3.3
# + Phase 6 wire-up), not the prior per-symbol nested shape.
# Keys: ``n_oos_trades``, ``n_executed``, ``n_rejected``,
# ``final_equity``, ``spa_p_value``, ``spa_naive_p_value``,
# ``spa_joint_k_trials``, ``hansen_fallback``. Documented
# in ``Candidate.run_edge_testing``.
for key, value in sorted(result.edge_metrics.items()):
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")
    else:
        print(f"  {key}: {value}")

## Next steps

- See `02_custom_experiment.ipynb` to register your own strategy.
- See `03_interpreting_results.ipynb` for SPA/PSR/FDR deep-dive.